# 03 — Build Silver

Build five Silver tables: the authoritative Madrid district reference, typed EMVS housing records, complete population and indicator histories, and validated district boundaries. Source geometry remains available alongside repaired mapping and analysis representations.

## 1. Runtime and table contracts

`silver_district_reference` contains one row for each of Madrid's 21 official districts. Boundary codes and names are authoritative, while repaired mapping and analysis geometry is published separately in `silver_district_boundaries`.

`silver_emvs_housing` excludes the preserved Bronze `TOTAL` record from its district grain, casts housing measures to integers, and reconciles their aggregates back to that record. Blank `RESTO PROGRAMAS` values remain null. The reporting window comes from the source preamble: 1 June 2015 through 30 April 2023.

`silver_population` retains all 1,071 observations from 2018 through 2024. It classifies city, district, and barrio grain from codes, parses population measures, and uses the district reference for canonical district names.

In [ ]:
import importlib
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

for module_name in [
    "urban_workflow.silver.table_operations",
    "urban_workflow.silver.geospatial_operations",
    "urban_workflow.silver.indicators",
    "urban_workflow.silver.population",
    "urban_workflow.silver.districts",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

from urban_workflow.silver.table_operations import (
    add_silver_metadata,
    build_district_reference,
    build_district_boundaries,
    build_emvs_housing,
    build_indicator_panel,
    build_population,
    validate_district_reference,
    validate_district_boundaries,
    validate_emvs_housing,
    validate_indicator_panel,
    validate_population,
    validate_persisted_silver_table,
    write_snapshot_table,
)

print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
if not all((CATALOG, SCHEMA)):
    raise ValueError("catalog and schema must be non-empty.")
BRONZE_EMVS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_emvs_housing"
BRONZE_BOUNDARIES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_district_boundaries"
BRONZE_POPULATION_TABLE = f"{CATALOG}.{SCHEMA}.bronze_population_district_barrio"
BRONZE_INDICATOR_TABLE = f"{CATALOG}.{SCHEMA}.bronze_district_indicator_panel"
SILVER_DISTRICT_TABLE = f"{CATALOG}.{SCHEMA}.silver_district_reference"
SILVER_EMVS_TABLE = f"{CATALOG}.{SCHEMA}.silver_emvs_housing"
SILVER_POPULATION_TABLE = f"{CATALOG}.{SCHEMA}.silver_population"
SILVER_INDICATOR_TABLE = f"{CATALOG}.{SCHEMA}.silver_indicator_panel"
SILVER_BOUNDARY_TABLE = f"{CATALOG}.{SCHEMA}.silver_district_boundaries"

silver_started_at = datetime.now(timezone.utc)
silver_run_id = (
    f"silver_{silver_started_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Silver run: {silver_run_id}")
print(f"District target: {SILVER_DISTRICT_TABLE}")
print(f"EMVS target: {SILVER_EMVS_TABLE}")
print(f"Population target: {SILVER_POPULATION_TABLE}")
print(f"Indicator target: {SILVER_INDICATOR_TABLE}")
print(f"Boundary target: {SILVER_BOUNDARY_TABLE}")

## 2. Canonical district reference

The official boundary table supplies the stable two-digit district code and canonical display name. Normalized keys remove accents, spaces, and punctuation solely for matching; downstream tables retain the canonical name.

In [ ]:
bronze_boundaries = spark.table(BRONZE_BOUNDARIES_TABLE)
district_reference = build_district_reference(bronze_boundaries)
district_validation = validate_district_reference(district_reference)
district_reference = add_silver_metadata(
    district_reference,
    silver_run_id=silver_run_id,
    processed_at_utc=silver_started_at,
)

print(district_validation)
display(
    district_reference.select(
        "district_code", "district_number", "district_name", "district_name_key"
    ).orderBy("district_code")
)

## 3. Typed and normalized EMVS housing

Most EMVS labels match after mechanical normalization. Three legacy labels use explicit, visible aliases: `MONCLOA` → Moncloa–Aravaca, `SAN BLAS` → San Blas–Canillejas, and `VALLECAS VILLA` → Villa de Vallecas. The validation gate requires complete one-to-one coverage and reconciliation to the Bronze total.

In [ ]:
bronze_emvs = spark.table(BRONZE_EMVS_TABLE)
silver_emvs = build_emvs_housing(bronze_emvs, district_reference)
emvs_validation = validate_emvs_housing(bronze_emvs, silver_emvs)
silver_emvs = add_silver_metadata(
    silver_emvs,
    silver_run_id=silver_run_id,
    processed_at_utc=silver_started_at,
)

print(emvs_validation)
display(
    silver_emvs.select(
        "district_code", "district_name", "district_name_source",
        "district_mapping_method", "housing_units_total",
        "housing_units_regulation", "housing_units_other_programs",
    ).orderBy("district_code")
)

## 4. Complete population history

Population remains long and detailed in Silver: seven annual city totals, 147 district-year records, and 917 barrio-year records. Geographic level is determined from codes, avoiding historical spacing differences in names. All non-city records join to the canonical district reference.

In [ ]:
bronze_population = spark.table(BRONZE_POPULATION_TABLE)
silver_population = build_population(bronze_population, district_reference)
population_validation = validate_population(bronze_population, silver_population)
silver_population = add_silver_metadata(
    silver_population,
    silver_run_id=silver_run_id,
    processed_at_utc=silver_started_at,
)

print(population_validation)
display(
    silver_population.select(
        "reference_date", "geographic_level", "district_code",
        "district_name", "barrio_code", "barrio_name",
        "population_total", "population_male", "population_female",
    ).orderBy(F.col("reference_date").desc(), "geographic_level", "district_code", "barrio_code")
)

## 5. Long indicator-panel history

All 116,930 observations remain in long form with separate panel and observation years, typed decimal values, normalized geography, and complete indicator hierarchy and source fields. Eight source-defined observation identities contain two different values each. Both values are retained and the resulting 16 rows are explicitly flagged for downstream review.

In [ ]:
bronze_indicators = spark.table(BRONZE_INDICATOR_TABLE)
silver_indicators = build_indicator_panel(bronze_indicators, district_reference)
indicator_validation = validate_indicator_panel(bronze_indicators, silver_indicators)
silver_indicators = add_silver_metadata(
    silver_indicators,
    silver_run_id=silver_run_id,
    processed_at_utc=silver_started_at,
)

print(indicator_validation)
display(
    silver_indicators.filter("has_conflicting_observations").select(
        "source_record_id", "observation_key", "panel_year",
        "observation_year", "district_code", "barrio_code",
        "indicator_name", "indicator_source",
        "indicator_value_raw", "indicator_value",
        "observation_group_count", "conflicting_value_count",
    ).orderBy("observation_key", "source_record_id")
)

## 6. Validated district boundaries

The 21 district geometries are small enough for an explicit driver-side Shapely repair. Source WKB remains unchanged. Invalid polygons pass through `make_valid`, non-area line remnants are removed, and clean WKB is stored in EPSG:4326 for mapping and EPSG:3035 for analysis. Repair evidence and projected area remain visible on every row.

In [ ]:
silver_boundaries, boundary_conversion = build_district_boundaries(
    spark,
    bronze_boundaries,
)
boundary_validation = validate_district_boundaries(
    silver_boundaries,
    district_reference,
)
silver_boundaries = add_silver_metadata(
    silver_boundaries,
    silver_run_id=silver_run_id,
    processed_at_utc=silver_started_at,
)

print(boundary_conversion)
print(boundary_validation)
display(
    silver_boundaries.filter("geometry_repaired").select(
        "district_code", "district_name",
        "source_geometry_validity_reason", "geometry_repair_method",
        "source_geometry_type", "geometry_type",
        "area_m2", "repair_area_relative_change",
    ).orderBy("district_code")
)

## 7. Persist and read back

All fixed snapshots use deterministic overwrite semantics. The read-back gate verifies row counts, required columns, and non-null Bronze and Silver lineage metadata.

In [ ]:
write_snapshot_table(district_reference, SILVER_DISTRICT_TABLE)
write_snapshot_table(silver_emvs, SILVER_EMVS_TABLE)
write_snapshot_table(silver_population, SILVER_POPULATION_TABLE)
write_snapshot_table(silver_indicators, SILVER_INDICATOR_TABLE)
write_snapshot_table(silver_boundaries, SILVER_BOUNDARY_TABLE)

spark.sql(
    f"COMMENT ON TABLE {SILVER_DISTRICT_TABLE} IS "
    "'Canonical non-geometric Madrid district reference derived from official boundaries.'"
)
spark.sql(
    f"COMMENT ON TABLE {SILVER_EMVS_TABLE} IS "
    "'Typed EMVS housing records normalized to official Madrid district codes.'"
)
spark.sql(
    f"COMMENT ON TABLE {SILVER_POPULATION_TABLE} IS "
    "'Typed Madrid population history at city, district, and barrio grain.'"
)
spark.sql(
    f"COMMENT ON TABLE {SILVER_INDICATOR_TABLE} IS "
    "'Typed long-form Madrid indicator history with explicit source-conflict flags.'"
)
spark.sql(
    f"COMMENT ON TABLE {SILVER_BOUNDARY_TABLE} IS "
    "'Validated Madrid district boundaries with source, mapping, and analysis WKB.'"
)

persisted_districts = validate_persisted_silver_table(
    spark,
    table_name=SILVER_DISTRICT_TABLE,
    expected_record_count=21,
    required_columns=["district_code", "district_number", "district_name", "district_name_key"],
)
persisted_emvs = validate_persisted_silver_table(
    spark,
    table_name=SILVER_EMVS_TABLE,
    expected_record_count=21,
    required_columns=[
        "district_code", "district_name", "reporting_start_date",
        "reporting_end_date", "housing_units_total",
        "housing_units_regulation", "housing_units_other_programs",
    ],
)
persisted_boundaries = validate_persisted_silver_table(
    spark,
    table_name=SILVER_BOUNDARY_TABLE,
    expected_record_count=21,
    required_columns=[
        "district_code", "district_name", "source_geometry_wkb",
        "source_geometry_was_valid", "geometry_repaired",
        "geometry_repair_method", "geometry_wkb", "geometry_crs",
        "analysis_geometry_wkb", "analysis_geometry_crs", "area_m2",
    ],
)
persisted_indicators = validate_persisted_silver_table(
    spark,
    table_name=SILVER_INDICATOR_TABLE,
    expected_record_count=116_930,
    required_columns=[
        "source_record_id", "observation_key", "panel_year",
        "observation_year", "observation_date", "geographic_level",
        "district_code", "barrio_code", "indicator_name",
        "indicator_value_raw", "indicator_value",
        "has_conflicting_observations",
    ],
)
persisted_population = validate_persisted_silver_table(
    spark,
    table_name=SILVER_POPULATION_TABLE,
    expected_record_count=1_071,
    required_columns=[
        "source_record_id", "reference_date", "reference_year",
        "geographic_level", "municipality_code", "district_code",
        "barrio_code", "population_total", "population_male",
        "population_female",
    ],
)

validate_district_reference(persisted_districts)
validate_emvs_housing(bronze_emvs, persisted_emvs)
validate_population(bronze_population, persisted_population)
validate_indicator_panel(bronze_indicators, persisted_indicators)
validate_district_boundaries(persisted_boundaries, persisted_districts)
print(
    f"Validated {persisted_districts.count()} districts, "
    f"{persisted_emvs.count()} EMVS records, "
    f"{persisted_population.count()} population records, and "
    f"{persisted_indicators.count()} indicator records, and "
    f"{persisted_boundaries.count()} district boundaries."
)
dbutils.notebook.exit(
    "SUCCESS: Validated 21 districts, 21 EMVS, 1,071 population, "
    "116,930 indicator, and 21 boundary records."
)

## 8. Slice status

Milestone 3 is complete. The district reference, EMVS, population, indicator, and boundary tables passed their live Databricks write and read-back gates. The boundary result contains 18 unchanged and three repaired geometries across all 21 canonical districts.